# Week 2 — Text Sentiment Measures
**Project:** Do LLMs Read 10-Ks Better Than Dictionaries?

| Track | Method | Tool |
|-------|--------|------|
| 1 | Loughran–McDonald (2011) word counts | `lm_dict` |
| 2 | FinBERT sentence-level neural sentiment | `ProsusAI/finbert` |
| 3 | Llama-3.1-8B-Instruct zero-shot tone score | Midway3 / vLLM |

**Input:** `s3://<bucket>/10k-project/processed/master_panel_test.parquet`  
**Output:** `s3://<bucket>/10k-project/processed/sentiment_scores.parquet`

In [ ]:
# Install dependencies (run once)
import subprocess, sys
pkgs = ["pyarrow", "tqdm", "requests", "transformers", "torch", "accelerate", "boto3"]
for p in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", p, "-q"], check=False)
print("✓ packages ready")

In [ ]:
import os, re, io, json, requests, warnings
from pathlib import Path
from tqdm.auto import tqdm

import numpy as np
import pandas as pd
import boto3

warnings.filterwarnings("ignore")

# ── Config ─────────────────────────────────────────────────────────────────
S3_BUCKET  = "yulinwang-10k-llm"
S3_PREFIX  = "10k-project"
TEST_MODE  = True   # set False for full S&P 1500 run

PANEL_KEY  = f"{S3_PREFIX}/processed/master_panel{'_test' if TEST_MODE else ''}.parquet"
LOCAL_TMP  = Path("/tmp/week2")
LOCAL_TMP.mkdir(exist_ok=True)

s3 = boto3.client("s3")
print(f"Bucket: {S3_BUCKET}")
print(f"Panel key: {PANEL_KEY}")

In [ ]:
# ── Load master panel from S3 ──────────────────────────────────────────────
obj = s3.get_object(Bucket=S3_BUCKET, Key=PANEL_KEY)
panel = pd.read_parquet(io.BytesIO(obj["Body"].read()))
print(f"Panel: {panel.shape[0]} firm-years, {panel.shape[1]} columns")
print(panel[["ticker","fyear","mda_len","rdq"]].to_string())

In [ ]:
# ── Load MD&A texts from S3 using s3_key column ────────────────────────────
def read_mda_from_s3(s3_key: str) -> str:
    if not s3_key:
        return ""
    try:
        obj = s3.get_object(Bucket=S3_BUCKET, Key=s3_key)
        return obj["Body"].read().decode("utf-8", errors="replace")
    except Exception as e:
        print(f"  ⚠ Failed to read {s3_key}: {e}")
        return ""

print("Loading MD&A texts from S3...")
panel["mda_text"] = panel["s3_key"].apply(read_mda_from_s3)
n_empty = (panel["mda_text"].str.len() == 0).sum()
print(f"✓ Loaded {len(panel)} MD&A texts  ({n_empty} empty)")
panel[["ticker","fyear","mda_len"]].assign(read_len=panel["mda_text"].str.len())

---
## Track 1 — Loughran–McDonald (2011) Dictionary
Key metric: `lm_net_sent = (positive − negative) / total_words`

In [ ]:
# ── Download LM Master Dictionary ──────────────────────────────────────────
LM_CSV = LOCAL_TMP / "LM_MasterDictionary.csv"

if not LM_CSV.exists():
    print("Downloading LM Master Dictionary...")
    url = ("https://sraf.nd.edu/wp-content/uploads/2022/01/"
           "LoughranMcDonald_MasterDictionary_1993-2021.csv")
    r = requests.get(url, timeout=60)
    LM_CSV.write_bytes(r.content)
    print(f"  Saved → {LM_CSV}")
else:
    print(f"Found cached copy: {LM_CSV}")

lm_raw = pd.read_csv(LM_CSV, low_memory=False)
print("Rows:", len(lm_raw))

In [ ]:
# ── Build word-set dictionaries ────────────────────────────────────────────
CATS = ["Negative","Positive","Uncertainty","Litigious","Strong_Modal","Weak_Modal"]

lm_sets = {}
for cat in CATS:
    lm_sets[cat] = set(lm_raw.loc[lm_raw[cat] > 0, "Word"].str.upper())
    print(f"  {cat:15s}: {len(lm_sets[cat]):5d} words")

print(f"\n  Total unique LM words: {len(set().union(*lm_sets.values()))}")

In [ ]:
# ── Tokenise & count ───────────────────────────────────────────────────────
_WORD_RE = re.compile(r"[A-Za-z]+")

def lm_counts(text: str) -> dict:
    tokens = [t.upper() for t in _WORD_RE.findall(text)]
    total  = len(tokens)
    if total == 0:
        return {f"lm_{c.lower()}": 0 for c in CATS} | {
            "lm_total_words": 0, "lm_net_sent": np.nan,
            "lm_tone": np.nan, "lm_uncertainty_pct": np.nan
        }
    counts = {f"lm_{c.lower()}": sum(1 for t in tokens if t in lm_sets[c])
              for c in CATS}
    pos = counts["lm_positive"]
    neg = counts["lm_negative"]
    unc = counts["lm_uncertainty"]
    counts["lm_total_words"]     = total
    counts["lm_net_sent"]        = (pos - neg) / total
    counts["lm_tone"]            = (pos - neg) / (pos + neg) if (pos + neg) > 0 else np.nan
    counts["lm_uncertainty_pct"] = unc / total
    return counts

lm_rows = []
for _, row in tqdm(panel.iterrows(), total=len(panel), desc="LM Dict"):
    lm_rows.append({"gvkey": row["gvkey"], "fyear": row["fyear"],
                    **lm_counts(row["mda_text"])})

lm_df = pd.DataFrame(lm_rows)
print("✓ Track 1 complete")
lm_df[["gvkey","fyear","lm_total_words","lm_net_sent","lm_tone","lm_uncertainty_pct"]].head(10)

---
## Track 2 — FinBERT Neural Sentiment
`fb_net` = mean(P(positive) − P(negative)) across 450-token chunks

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

print("Loading FinBERT...")
fb_tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
fb_model     = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
fb_model.eval().to(DEVICE)
print(f"✓ FinBERT ready  (labels: {fb_model.config.id2label})")

In [ ]:
def chunk_text(text: str, chunk_tokens=450, stride_tokens=50) -> list:
    words = text.split()
    step   = int(chunk_tokens / 1.3)
    stride = int(stride_tokens / 1.3)
    chunks, i = [], 0
    while i < len(words):
        chunk = " ".join(words[i: i + step])
        if chunk.strip():
            chunks.append(chunk)
        i += step - stride
    return chunks if chunks else [text[:2000]]

@torch.no_grad()
def score_chunks(chunks: list) -> np.ndarray:
    probs_all = []
    for i in range(0, len(chunks), 16):
        batch  = chunks[i: i + 16]
        enc    = fb_tokenizer(batch, padding=True, truncation=True,
                              max_length=512, return_tensors="pt").to(DEVICE)
        logits = fb_model(**enc).logits
        probs_all.append(F.softmax(logits, dim=-1).cpu().numpy())
    return np.vstack(probs_all)

def finbert_doc_score(text: str) -> dict:
    if not text.strip():
        return {"fb_positive": np.nan, "fb_negative": np.nan,
                "fb_neutral": np.nan, "fb_net": np.nan, "fb_n_chunks": 0}
    label_map = {v: k for k, v in fb_model.config.id2label.items()}
    pos_idx, neg_idx, neu_idx = label_map["positive"], label_map["negative"], label_map["neutral"]
    chunks = chunk_text(text)
    probs  = score_chunks(chunks)
    return {
        "fb_positive":  float(probs[:, pos_idx].mean()),
        "fb_negative":  float(probs[:, neg_idx].mean()),
        "fb_neutral":   float(probs[:, neu_idx].mean()),
        "fb_net":       float((probs[:, pos_idx] - probs[:, neg_idx]).mean()),
        "fb_n_chunks":  len(chunks)
    }

fb_rows = []
for _, row in tqdm(panel.iterrows(), total=len(panel), desc="FinBERT"):
    fb_rows.append({"gvkey": row["gvkey"], "fyear": row["fyear"],
                    **finbert_doc_score(row["mda_text"])})

fb_df = pd.DataFrame(fb_rows)
print("✓ Track 2 complete")
fb_df.head(10)

---
## Track 3 — Llama-3.1-8B-Instruct (Midway3)

This section builds JSONL batch files and uploads them to S3.  
Midway3 reads them directly from S3, scores each MD&A, and writes results back to S3.

Four scoring dimensions (0–10):
- `management_optimism`: overall tone of management language
- `guidance_specificity`: concrete numerical guidance vs. vague qualitative language
- `uncertainty_hedging`: degree of hedging / uncertainty language
- `risk_framing`: severity of disclosed risks

In [ ]:
SYSTEM_PROMPT = """\
You are a financial economist analyzing a 10-K MD&A section.
Output format (integers 0-10):
{"management_optimism": <int>, "guidance_specificity": <int>, "uncertainty_hedging": <int>, "risk_framing": <int>}

Definitions:
- management_optimism:  0=very pessimistic, 5=neutral, 10=very optimistic
- guidance_specificity: 0=vague qualitative only, 10=detailed numerical guidance (EPS, revenue targets)
- uncertainty_hedging:  0=highly confident language, 10=extreme hedging/uncertainty
- risk_framing:         0=no material risks, 10=severe/numerous risks disclosed

Respond ONLY with the JSON object. No explanation, no markdown."""

def build_messages(mda_text: str, max_chars: int = 6000) -> list:
    excerpt = mda_text[:max_chars].strip()
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"MD&A excerpt:\n\n{excerpt}"}
    ]

sample = panel.iloc[0]
msgs   = build_messages(sample["mda_text"])
print(f"Filing: {sample['ticker']} FY{sample['fyear']}")
print(f"User message length: {len(msgs[1]['content'])} chars")

In [ ]:
# ── Write JSONL shards and upload to S3 ────────────────────────────────────
SHARD_SIZE  = 20
BATCH_PREFIX = f"{S3_PREFIX}/llm_batches"

records = []
for _, row in panel.iterrows():
    records.append({
        "id":           f"{row['gvkey']}_{row['fyear']}",
        "gvkey":        row["gvkey"],
        "ticker":       row["ticker"],
        "fyear":        int(row["fyear"]),
        "accession_no": row.get("accession_no", ""),
        "messages":     build_messages(row["mda_text"])
    })

shard_keys = []
for i in range(0, len(records), SHARD_SIZE):
    shard_id = i // SHARD_SIZE
    lines = "\n".join(json.dumps(r) for r in records[i: i + SHARD_SIZE])
    key   = f"{BATCH_PREFIX}/batch_{shard_id:03d}.jsonl"
    s3.put_object(Bucket=S3_BUCKET, Key=key, Body=lines.encode("utf-8"))
    shard_keys.append(key)
    print(f"  ✓ Uploaded {key}  ({len(records[i: i + SHARD_SIZE])} records)")

print(f"\n✓ {len(shard_keys)} shard(s) uploaded to s3://{S3_BUCKET}/{BATCH_PREFIX}/")
print(f"\nNext: on Midway3, run:")
print(f"  aws s3 cp s3://{S3_BUCKET}/{BATCH_PREFIX}/ $SCRATCH/llm_batches/ --recursive")
print(f"  sbatch --constraint=a100 submit_llama.sh")

---
### 3B. Load Llama Results (run after Midway3 job completes)

After SLURM jobs finish, copy results back to S3:
```bash
# On Midway3:
aws s3 cp $SCRATCH/llm_out/ s3://yulinwang-10k-llm/10k-project/llm_results/ --recursive
```
Then run the cell below.

In [ ]:
# ── Parse Llama outputs from S3 ────────────────────────────────────────────
RESULTS_PREFIX = f"{S3_PREFIX}/llm_results"

def parse_llm_json(raw: str) -> dict | None:
    raw = raw.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass
    m = re.search(r"\{[^{}]+\}", raw, re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except json.JSONDecodeError:
            pass
    return None

llm_rows = []
try:
    resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=RESULTS_PREFIX)
    result_keys = [o["Key"] for o in resp.get("Contents", []) if o["Key"].endswith(".jsonl")]
    print(f"Found {len(result_keys)} result file(s) in S3")
    for key in result_keys:
        obj  = s3.get_object(Bucket=S3_BUCKET, Key=key)
        for line in obj["Body"].read().decode().splitlines():
            rec    = json.loads(line)
            parsed = parse_llm_json(rec.get("output", ""))
            if parsed:
                llm_rows.append({
                    "gvkey":            rec["gvkey"],
                    "fyear":            rec["fyear"],
                    "llm_mgmt_opt":     parsed.get("management_optimism"),
                    "llm_guidance":     parsed.get("guidance_specificity"),
                    "llm_uncertainty":  parsed.get("uncertainty_hedging"),
                    "llm_risk":         parsed.get("risk_framing"),
                })
except Exception as e:
    print(f"⚠ Could not load Llama results: {e}")

llm_df = pd.DataFrame(llm_rows)
if len(llm_df):
    print(f"✓ Parsed {len(llm_df)} Llama scores")
    print(llm_df.head())
else:
    print("⚠ No Llama results yet — run Midway3 job first, then re-run this cell.")

---
## Merge All Tracks → Final Panel

In [ ]:
merged = (
    panel
    .drop(columns=["mda_text"], errors="ignore")
    .merge(lm_df, on=["gvkey","fyear"], how="left")
    .merge(fb_df, on=["gvkey","fyear"], how="left")
)
if not llm_df.empty:
    merged = merged.merge(llm_df, on=["gvkey","fyear"], how="left")

print(f"Merged panel shape: {merged.shape}")
print("Columns:", merged.columns.tolist())

In [ ]:
# ── Cross-sectional summary ────────────────────────────────────────────────
summary = merged[["ticker","fyear","lm_net_sent","lm_tone","lm_uncertainty_pct",
                   "fb_net","lm_total_words"]].sort_values(["ticker","fyear"])
print(summary.to_string(index=False))

# ── Correlation table ──────────────────────────────────────────────────────
text_cols = ["lm_net_sent","lm_tone","lm_uncertainty_pct","fb_net","fb_positive","fb_negative"]
if not llm_df.empty:
    text_cols += ["llm_mgmt_opt","llm_guidance","llm_uncertainty","llm_risk"]
print("\nCorrelation matrix:")
print(merged[text_cols].corr().round(3).to_string())

In [ ]:
# ── Save to S3 ─────────────────────────────────────────────────────────────
buf = io.BytesIO()
merged.to_parquet(buf, index=False)
buf.seek(0)
out_key = f"{S3_PREFIX}/processed/sentiment_scores{'_test' if TEST_MODE else ''}.parquet"
s3.put_object(Bucket=S3_BUCKET, Key=out_key, Body=buf.read())
print(f"✓ Saved → s3://{S3_BUCKET}/{out_key}")
print(f"  Shape: {merged.shape}")